# Homework 3
### 1(a) Download AReM Data
Establish Path to Data Files

In [2]:
from pathlib import Path
import pandas as pd

data_dir = Path("AReM")

print("Notebook working directory:", Path.cwd())
print("Data folder exists:", data_dir.is_dir())

files = sorted(data_dir.glob("*/dataset*.csv"))
print("Total CSV files:", len(files))

for folder in sorted(p for p in data_dir.iterdir() if p.is_dir()):
    print(f"{folder.name}: {len(list(folder.glob('dataset*.csv')))} files")

Notebook working directory: /Users/bt/Documents/DSCI 552/Homework 3
Data folder exists: True
Total CSV files: 88
bending1: 7 files
bending2: 6 files
cycling: 15 files
lying: 15 files
sitting: 15 files
standing: 15 files
walking: 15 files


### 1(b) Choosing Datasets
Keep datasets 1 and 2 in folders bending1 and bending 2, as well as datasets 1,
2, and 3 in other folders as test data and other datasets as train data.

In [3]:
records = []

for path in files:
    activity = path.parent.name
    dataset_number = int(path.stem.removeprefix("dataset"))

    test_limit = 2 if activity in {"bending1", "bending2"} else 3
    split = "test" if dataset_number <= test_limit else "train"

    records.append({
        "activity": activity,
        "dataset_number": dataset_number,
        "split": split,
        "path": path,
    })

manifest = pd.DataFrame(records)

display(
    manifest.groupby(["activity", "split"])
    .size()
    .unstack(fill_value=0)
)

print(manifest["split"].value_counts())

split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


split
train    69
test     19
Name: count, dtype: int64


# Check for cleaning
Since some of the files need minor cleaning, I inspected for problem files

In [24]:
from collections import Counter

problems = []

for path in files:
    with path.open(encoding="utf-8-sig") as handle:
        rows = [
            line.strip()
            for line in handle
            if line.strip() and not line.lstrip().startswith("#")
        ]

    field_counts = Counter(row.count(",") + 1 for row in rows)

    if len(rows) != 480 or field_counts != Counter({7: 480}):
        problems.append((str(path), len(rows), dict(field_counts)))

print("Files needing inspection:", len(problems))
for problem in problems:
    print(problem)

Files needing inspection: 4
('AReM/bending2/dataset4.csv', 480, {1: 480})
('AReM/cycling/dataset14.csv', 480, {7: 479, 8: 1})
('AReM/cycling/dataset9.csv', 480, {7: 479, 8: 1})
('AReM/sitting/dataset8.csv', 479, {7: 479})


# Observe Errors
Look at format errors and other issues regarding the cleaning errors

In [25]:
problem_paths = [
    data_dir / "bending2/dataset4.csv",
    data_dir / "cycling/dataset14.csv",
    data_dir / "cycling/dataset9.csv",
    data_dir / "sitting/dataset8.csv",
]

for path in problem_paths:
    with path.open(encoding="utf-8-sig") as handle:
        rows = [
            (line_number, line.strip())
            for line_number, line in enumerate(handle, start=1)
            if line.strip() and not line.lstrip().startswith("#")
        ]

    print(f"\n{path}")
    print("First two:", rows[:2])
    print("Last two:", rows[-2:])

    unusual = [
        i for i, (_, text) in enumerate(rows)
        if text.count(",") + 1 != 7
    ]

    # Show nearby rows for a few format errors, not all 480.
    for i in unusual[:2]:
        print("Around unusual row:")
        for nearby in rows[max(0, i - 1):i + 2]:
            print(nearby)

    if path.name == "dataset8.csv" and path.parent.name == "sitting":
        times = [float(text.split(",")[0]) for _, text in rows]
        steps = [b - a for a, b in zip(times, times[1:])]
        print("Time-step counts:", __import__("collections").Counter(steps))


AReM/bending2/dataset4.csv
First two: [(6, '0 32.50 0.50 0.00 0.00 19.00 1.00'), (7, '250 32.50 0.50 0.00 0.00 18.50 0.50')]
Last two: [(484, '119500 28.00 0.00 5.00 0.82 17.00 0.71'), (485, '119750 28.00 0.00 0.00 0.00 17.00 1.00')]
Around unusual row:
(6, '0 32.50 0.50 0.00 0.00 19.00 1.00')
(7, '250 32.50 0.50 0.00 0.00 18.50 0.50')
Around unusual row:
(6, '0 32.50 0.50 0.00 0.00 19.00 1.00')
(7, '250 32.50 0.50 0.00 0.00 18.50 0.50')
(8, '500 32.75 0.43 1.00 0.00 18.00 0.00')

AReM/cycling/dataset14.csv
First two: [(6, '0,35.75,1.79,21.00,2.45,20.50,0.50'), (7, '250,38.75,1.48,17.00,4.32,19.67,2.05')]
Last two: [(484, '119500,33.33,6.18,15.50,3.77,17.00,4.32'), (485, '119750,34.00,2.12,24.50,1.80,22.25,2.59,')]
Around unusual row:
(484, '119500,33.33,6.18,15.50,3.77,17.00,4.32')
(485, '119750,34.00,2.12,24.50,1.80,22.25,2.59,')

AReM/cycling/dataset9.csv
First two: [(6, '0,41.67,0.47,17.67,2.87,14.50,2.06'), (7, '250,42.00,2.12,9.67,6.34,14.75,2.49')]
Last two: [(484, '119500,34.2

# Clean errors
Ignore columns and remove empty fields cause by trailing commas

In [26]:
sensor_cols = [
    "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23",
]
expected_time = pd.Index(range(0, 120000, 250), name="time")


def load_instance(path):
    rows = []

    with path.open(encoding="utf-8-sig") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()

            if not line or line.startswith("#"):
                continue

            parts = line.split(",") if "," in line else line.split()

            # Remove an extra empty field caused by a trailing comma.
            if len(parts) == 8 and parts[-1] == "":
                parts = parts[:-1]

            if len(parts) != 7:
                raise ValueError(
                    f"{path}, line {line_number}: expected 7 fields, got {len(parts)}"
                )

            rows.append([float(value) for value in parts])

    frame = pd.DataFrame(rows, columns=["time", *sensor_cols])
    frame["time"] = frame["time"].astype(int)

    if not frame["time"].is_unique:
        raise ValueError(f"Duplicate times in {path}")

    missing = expected_time.difference(frame["time"])
    extra = pd.Index(frame["time"]).difference(expected_time)

    if len(extra):
        raise ValueError(f"Unexpected times in {path}: {list(extra)}")

    if len(missing):
        if path.parent.name == "sitting" and path.name == "dataset8.csv" and len(missing) == 1:
            print(f"Interpolating missing time {missing[0]} in {path}")
            frame = frame.set_index("time").reindex(expected_time)
            frame[sensor_cols] = frame[sensor_cols].interpolate(
                method="index", limit_area="inside"
            )
            frame = frame.reset_index()
        else:
            raise ValueError(f"Unexpected missing times in {path}: {list(missing)}")

    if len(frame) != 480 or frame[sensor_cols].isna().any().any():
        raise ValueError(f"Incomplete data after cleaning: {path}")

    return frame


instances = {path: load_instance(path) for path in files}

print("Instances loaded:", len(instances))
print("Rows per instance:", set(len(frame) for frame in instances.values()))

Interpolating missing time 13500 in AReM/sitting/dataset8.csv
Instances loaded: 88
Rows per instance: {480}


### 1(c)i Standard Features in Time-Series Classification
List: min, mean, median, max, std, 1st quartile, 2nd quartile,
also other common features are skewness and RMS

In [27]:
feature_rows = []

for row in manifest.itertuples(index=False):
    frame = instances[row.path]

    result = {
        "instance": len(feature_rows) + 1,
        "file": str(row.path),
        "activity": row.activity,
        "split": row.split,
    }

    ### 1(c)ii
    # Extract the time-domain features minimum, maximum, mean, median, stan-dard deviation,
    # first quartile, and third quartile for all of the 6 time series in each instance.
    for i, column in enumerate(sensor_cols, start=1):
        series = frame[column]

        result[f"min{i}"] = series.min()
        result[f"max{i}"] = series.max()
        result[f"mean{i}"] = series.mean()
        result[f"median{i}"] = series.median()
        result[f"std{i}"] = series.std(ddof=1)
        result[f"q1_{i}"] = series.quantile(0.25)
        result[f"q3_{i}"] = series.quantile(0.75)

    feature_rows.append(result)

features_df = pd.DataFrame(feature_rows)

metadata_cols = ["instance", "file", "activity", "split"]
feature_cols = [col for col in features_df if col not in metadata_cols]

print("Table shape:", features_df.shape)
print("Numeric feature columns:", len(feature_cols))
print("Missing feature values:", features_df[feature_cols].isna().sum().sum())
display(features_df.head())

Table shape: (88, 46)
Numeric feature columns: 42
Missing feature values: 0


,instance,file,activity,split,min1,max1,mean1,median1,std1,q1_1,...,std5,q1_5,q3_5,min6,max6,mean6,median6,std6,q1_6,q3_6
0,1,AReM/bending1/dataset1.csv,bending1,test,37.25,45.00,40.624792,40.50,1.476967,39.25,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,2,AReM/bending1/dataset2.csv,bending1,test,38.00,45.67,42.812812,42.50,1.435550,42.00,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,3,AReM/bending1/dataset3.csv,bending1,train,35.00,47.40,43.954500,44.33,1.558835,43.00,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,4,AReM/bending1/dataset4.csv,bending1,train,33.00,47.75,42.179812,43.50,3.670666,39.15,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,5,AReM/bending1/dataset5.csv,bending1,train,33.00,45.75,41.678063,41.75,2.243490,41.33,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


### 1(c)iii Standard Deviation and Bootstrap

In [28]:
import numpy as np
from scipy.stats import bootstrap

def sample_std(values, axis=-1):
    return np.std(values, axis=axis, ddof=1)

rng = np.random.default_rng(552)
results = []

for column in feature_order:
    values = features_df[column].to_numpy(dtype=float)

    boot = bootstrap(
        (values,),
        sample_std,
        vectorized=True,
        confidence_level=0.90,
        n_resamples=10_000,
        method="percentile",
        rng=rng,
    )

    results.append({
        "feature": quartile_names.get(column, column),
        "std_estimate": sample_std(values),
        "ci_lower": boot.confidence_interval.low,
        "ci_upper": boot.confidence_interval.high,
    })

### 1(c)iii Top 3 Features
I selected mean, standard deviation, and maximum because they describe different aspects of each time series. Mean summarizes the overall level of the readings, while standard deviation shows how much they fluctuate during an activity. Maximum captures the highest reading reached. I chose mean over median because both describe a typical level, but mean reflects the magnitude of every reading. I chose maximum over minimum because the minimum features for series 2 and 4 are constant across all 88 instances, while their maximum features vary. This makes the three features a reasonable choice, although their ability to distinguish activities would need to be tested separately.

In [29]:
std_ci_df = pd.DataFrame(results)

selected_columns = [
    f"{feature_type}{i}"
    for feature_type in ("mean", "std", "max")
    for i in range(1, 7)
]

selected_ci_df = (
    std_ci_df.set_index("feature")
    .loc[selected_columns]
    .reset_index()
)

print("Selected features:", len(selected_ci_df))
display(selected_ci_df.round(4))

print("All features:", len(std_ci_df))
display(std_ci_df.round(4))

Selected features: 18


,feature,std_estimate,ci_lower,ci_upper
0,mean1,5.3357,4.7128,5.8836
1,mean2,1.5742,1.3908,1.7068
2,mean3,4.0084,3.4124,4.4867
3,mean4,1.1661,1.0775,1.2225
4,mean5,5.6756,4.4258,6.7405
5,mean6,1.1548,1.0631,1.2157
6,std1,1.7722,1.5726,1.9494
7,std2,0.8841,0.8032,0.9420
8,std3,0.9467,0.7618,1.1219
9,std4,0.4582,0.4203,0.4856


All features: 42


,feature,std_estimate,ci_lower,ci_upper
0,min1,9.5700,8.2569,10.7417
1,max1,4.3944,3.3280,5.3082
2,mean1,5.3357,4.7128,5.8836
3,median1,5.4401,4.8092,5.9993
4,std1,1.7722,1.5726,1.9494
5,1st quart1,6.1536,5.5706,6.6526
6,3rd quart1,5.1389,4.3307,5.8409
7,min2,0.0000,0.0000,0.0000
8,max2,5.0627,4.6327,5.4113
9,mean2,1.5742,1.3908,1.7068


# 2. ISLR 3.7.4
a. The cubic model’s training RSS will be lower or the same. It can always fit the training data at least as well as the linear model because it has extra terms. Although its more likely that it is lower, and impossible that it is higher. We know this because both X squared and X cubed coefficients can be 0 therefore it is linear.

b. I would expect the linear model’s test RSS to be lower because the real relationship is linear. The cubic model might learn random patterns in the training data that do not appear in new data.

c. The cubic model’s training RSS will be lower or the same. This should be true even when the real relationship is not linear because the cubic RSS for the same coefficient reasons as question a, the coefficients should either help fit the training data which means the cubic model would be lower or not help fit the training data which would make it the same.

d. We cannot tell which model will have lower test RSS. The cubic model might capture the curve better, but it might also fit random patterns that do not help with new data.